# Malaika — Fine-Tune Gemma 4 for Breath Sounds (v4)

**v4 fixes:**
1. **Simplified output**: single word (normal/crackle/wheeze/both) instead of complex JSON
2. **Colormap spectrograms**: viridis colormap instead of grayscale (more visual info)
3. **Lower learning rate**: 5e-5 (was 2e-4) — prevents jumping to class bias
4. **More steps**: 500 (was 300) — slower learning needs more time
5. **Class-balanced** oversampling (kept from v3)

| | v1 | v2 | v3 | v4 (this) |
|--|--|--|--|--|
| Output | JSON | JSON | JSON | **Single word** |
| Spectrogram | grayscale | grayscale | grayscale | **viridis colormap** |
| LR | 2e-4 | 2e-4 | 2e-4 | **5e-5** |
| Steps | 100 | 300 | 300 | **500** |
| Balanced | No | No | Yes | Yes |
| Accuracy | 20% | 50% | 20% | TBD |

In [ ]:
# Cell 1: Install
!pip install -q git+https://github.com/huggingface/transformers.git peft trl datasets bitsandbytes accelerate librosa Pillow soundfile kaggle matplotlib

In [ ]:
# Cell 2: Auth + imports
from huggingface_hub import login
import os, subprocess, json, random, re, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import librosa
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret("HF_TOKEN"))
    KU, KK = secrets.get_secret("KAGGLE_USERNAME"), secrets.get_secret("KAGGLE_KEY")
    print("Auth: Kaggle")
except ModuleNotFoundError:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
        KU, KK = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        print("Auth: Colab")
    except Exception:
        login()
        KU, KK = os.environ.get("KAGGLE_USERNAME", ""), os.environ.get("KAGGLE_KEY", "")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KU, "key": KK}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")

In [ ]:
# Cell 3: Download ICBHI
ICBHI_DIR = Path("/tmp/icbhi_data")
ICBHI_INNER = ICBHI_DIR / "Respiratory_Sound_Database" / "Respiratory_Sound_Database"
KAGGLE_NATIVE = Path("/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database")

if KAGGLE_NATIVE.exists():
    audio_dir = KAGGLE_NATIVE / "audio_and_txt_files"
elif ICBHI_INNER.exists():
    audio_dir = ICBHI_INNER / "audio_and_txt_files"
else:
    print("Downloading ICBHI...")
    ICBHI_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "vbookshelf/respiratory-sound-database",
        "-p", str(ICBHI_DIR), "--unzip"], capture_output=True, text=True, check=True)
    audio_dir = ICBHI_INNER / "audio_and_txt_files"

audio_files = list(audio_dir.glob("*.wav"))
print(f"ICBHI: {len(audio_files)} audio files")

In [ ]:
# Cell 4: Parse annotations
def parse_icbhi_annotations(adir):
    records = []
    for txt in sorted(adir.glob("*.txt")):
        wav = txt.with_suffix(".wav")
        if not wav.exists(): continue
        pid = txt.stem.split("_")[0]
        hc, hw = False, False
        with open(txt) as f:
            for line in f:
                ff = line.strip().split()
                if len(ff) >= 4:
                    if int(ff[2]) == 1: hc = True
                    if int(ff[3]) == 1: hw = True
        label = "both" if hc and hw else "crackle" if hc else "wheeze" if hw else "normal"
        records.append({"audio_path": str(wav), "patient_id": pid,
            "label": label, "has_crackle": hc, "has_wheeze": hw})
    return records

records = parse_icbhi_annotations(audio_dir)
print(f"Parsed {len(records)} recordings:")
for label, count in sorted(Counter(r["label"] for r in records).items()):
    print(f"  {label}: {count}")

In [ ]:
# Cell 5: Generate COLORMAP spectrograms (v4 fix — viridis instead of grayscale)
SPEC_DIR = Path("/tmp/spectrograms_v4")
SPEC_DIR.mkdir(exist_ok=True)
SPEC_W, SPEC_H = 512, 256

# Get viridis colormap
viridis = plt.cm.get_cmap('viridis')

def audio_to_spec_color(audio_path, output_path):
    """Convert audio to colormap spectrogram (more visual info than grayscale)."""
    try:
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
        if len(y) == 0: return False
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=50, fmax=4000)
        db = librosa.power_to_db(mel, ref=np.max)
        mn, mx = db.min(), db.max()
        norm = (db - mn) / (mx - mn) if mx > mn else np.zeros_like(db)
        norm = np.flip(norm, axis=0)  # low freq at bottom
        # Apply viridis colormap -> RGB
        colored = (viridis(norm)[:, :, :3] * 255).astype(np.uint8)
        Image.fromarray(colored).resize((SPEC_W, SPEC_H), Image.Resampling.LANCZOS).save(str(output_path))
        return True
    except: return False

print("Generating colormap spectrograms...")
spec_records = []
for i, r in enumerate(records):
    sp = SPEC_DIR / f"{Path(r['audio_path']).stem}_spec.png"
    if sp.exists() or audio_to_spec_color(r['audio_path'], sp):
        r["spectrogram_path"] = str(sp)
        spec_records.append(r)
    if (i+1) % 200 == 0: print(f"  {i+1}/{len(records)}")
records = spec_records
print(f"Generated {len(records)} spectrograms")

# Show a sample
sample_img = Image.open(records[0]["spectrogram_path"])
print(f"Sample: {records[0]['label']} — {sample_img.size} {sample_img.mode}")

In [ ]:
# Cell 6: SIMPLIFIED instruction pairs + balanced split
# v4: Single-word output instead of JSON — much easier for model to learn

INSTRUCTION = ("This is a mel-spectrogram (viridis colormap) of a child's breathing audio.\n"
    "Frequency: 50-4000 Hz (bottom to top). Time: left to right. Color: intensity.\n\n"
    "Classify the breath sounds into exactly ONE category:\n"
    "- normal: even, low-intensity pattern, no prominent bands or spots\n"
    "- wheeze: continuous horizontal bright bands (200-1000 Hz range)\n"
    "- crackle: short vertical bright spots, discontinuous, scattered\n"
    "- both: wheeze bands AND crackle spots present together\n\n"
    "Respond with ONLY the category name, nothing else.")

pairs = [{"spectrogram_path": r["spectrogram_path"], "label": r["label"],
    "instruction": INSTRUCTION,
    "response": r["label"]} for r in records]  # Simple single-word response!

# Patient-level split
pids = list(set(r["patient_id"] for r in records))
random.seed(42); random.shuffle(pids)
train_pats = set(pids[:int(len(pids)*0.8)])
train_pairs_raw = [p for p, r in zip(pairs, records) if r["patient_id"] in train_pats]
test_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] not in train_pats]

# Class balancing
by_label = defaultdict(list)
for p in train_pairs_raw: by_label[p["label"]].append(p)
print("Before balancing:")
for l, items in sorted(by_label.items()): print(f"  {l}: {len(items)}")

max_count = max(len(v) for v in by_label.values())
train_pairs = []
for label, items in by_label.items():
    oversampled = (items * (max_count // len(items) + 1))[:max_count]
    train_pairs.extend(oversampled)
random.seed(42); random.shuffle(train_pairs)

print(f"\nAfter balancing:")
for l, c in sorted(Counter(p["label"] for p in train_pairs).items()): print(f"  {l}: {c}")
print(f"\nTrain: {len(train_pairs)} | Test: {len(test_pairs)}")
print(f"Test: {dict(Counter(p['label'] for p in test_pairs))}")
print(f"\nSample response: '{train_pairs[0]['response']}'  (was complex JSON before)")

In [ ]:
# Cell 7: Load model
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E4B-it"
print(f"Loading {MODEL_NAME}...")
t0 = time.monotonic()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"),
    torch_dtype=torch.float16,
)
print(f"Loaded in {time.monotonic()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 8: Add LoRA adapter
from peft import LoraConfig, get_peft_model, TaskType

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

unwrapped = 0
for name, module in list(model.named_modules()):
    if module.__class__.__name__ == "Gemma4ClippableLinear":
        parts = name.split(".")
        parent = model
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        unwrapped += 1
print(f"Unwrapped {unwrapped} layers")

model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.1, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 9: Build dataset (simplified single-word output)
from torch.utils.data import Dataset as TorchDataset

SYSTEM_MSG = ("You are a breath sound classifier. Look at the spectrogram image carefully. "
    "Respond with exactly one word: normal, wheeze, crackle, or both.")

class SpectrogramDataset(TorchDataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": spec_img},
                {"type": "text", "text": f"{SYSTEM_MSG}\n\n{pair['instruction']}"},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": pair["response"]},
            ]},
        ]
        inputs = processor.apply_chat_template(
            messages, tokenize=True, return_dict=True,
            return_tensors="pt", add_generation_prompt=False,
        )
        input_ids = inputs["input_ids"].squeeze(0)
        labels = input_ids.clone()
        # Mask everything except the response (just 1-2 tokens now!)
        resp_len = len(tokenizer.encode(pair["response"], add_special_tokens=False))
        if resp_len > 0 and len(labels) > resp_len:
            labels[:-resp_len] = -100
        result = {"labels": labels}
        for key in inputs:
            if inputs[key] is not None:
                result[key] = inputs[key].squeeze(0)
        return result

train_dataset = SpectrogramDataset(train_pairs)
sample = train_dataset[0]
print(f"Dataset: {len(train_dataset)} examples")
# Show how few tokens the response is now
resp_tokens = tokenizer.encode(train_pairs[0]["response"], add_special_tokens=False)
print(f"Response '{train_pairs[0]['response']}' = {len(resp_tokens)} token(s) (was ~50 tokens for JSON)")
print("Fields:")
for k, v in sample.items():
    if hasattr(v, 'shape'): print(f"  {k}: {v.shape}")

In [ ]:
# Cell 10: Train (v4: 500 steps, lower LR)
from transformers import TrainingArguments, Trainer

def gemma4_collator(features):
    batch = {}
    for key in features[0].keys():
        values = [f[key] for f in features if key in f and f[key] is not None]
        if not values: continue
        if isinstance(values[0], torch.Tensor):
            try: batch[key] = torch.stack(values)
            except RuntimeError: batch[key] = values[0].unsqueeze(0)
        else: batch[key] = values
    return batch

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./breath_sound_lora_v4",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=500,
        learning_rate=5e-5,
        warmup_steps=30,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        seed=42,
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    ),
    train_dataset=train_dataset,
    data_collator=gemma4_collator,
)

print("v4: 500 steps, lr=5e-5 cosine, single-word output, colormap spectrograms")
print(f"Dataset: {len(train_dataset)} balanced examples")
print(f"Estimated: ~75 min on T4")
t0 = time.monotonic()
result = trainer.train()
train_time = time.monotonic() - t0
print(f"\nDone in {train_time/60:.1f} min | Loss: {result.training_loss:.4f}")

In [ ]:
# Cell 11: Save adapter
ADAPTER_NAME = "malaika-breath-sounds-lora-v4"
model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Saved to {ADAPTER_NAME}/")
for f in sorted(Path(ADAPTER_NAME).glob("*")):
    if f.is_file(): print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f} MB")

In [ ]:
# Cell 12: Evaluate
model.gradient_checkpointing_disable()
model.eval()

print("=" * 60)
print("EVALUATION ON TEST SET (v4 — single-word classification)")
print("=" * 60)

correct, total_test = 0, 0
per_label = {"normal": [0,0], "wheeze": [0,0], "crackle": [0,0], "both": [0,0]}

for pair in test_pairs[:40]:  # evaluate more samples
    spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": spec_img},
        {"type": "text", "text": pair["instruction"]}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    )
    inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    
    label = pair["label"]
    per_label[label][1] += 1
    
    # Match prediction to one of the 4 classes
    pred_label = "unknown"
    for candidate in ["both", "crackle", "wheeze", "normal"]:  # check 'both' first
        if candidate in pred_text:
            pred_label = candidate
            break
    
    if pred_label == label:
        correct += 1; per_label[label][0] += 1; status = "PASS"
    else:
        status = f"FAIL (pred={pred_label})"
    total_test += 1
    print(f"  {status:35s} [{label:>7s}] -> '{pred_text[:30]}'")

print(f"\nOverall: {correct}/{total_test} ({100*correct/total_test:.0f}%)")
print(f"Baseline: 25% | v1: 20% | v2: 50% | v3: 20%")
print(f"\nPer-label:")
for label, (c, t) in per_label.items():
    if t > 0: print(f"  {label:>8s}: {c}/{t} ({100*c/t:.0f}%)")

In [ ]:
# Cell 13: Summary
print("=" * 60)
print("FINE-TUNING RESULTS — ALL VERSIONS")
print("=" * 60)
print(f"""
## Breath Sound Classification via Spectrogram Vision Fine-Tuning

### Innovation
Gemma 4 cannot process audio natively. We convert breath sounds to
mel-spectrogram images and fine-tune vision to classify them.

### Results Progression
| Version | Config | Accuracy | Key Change |
|---------|--------|----------|------------|
| Baseline | No fine-tuning | 25% | Predicts all normal |
| v1 | r=8, JSON, 100 steps | 20% | LoRA too small |
| v2 | r=32, JSON, 300 steps | 50% | Learned crackle only |
| v3 | v2 + balanced | 20% | Swung to all normal |
| v4 | Single-word, colormap, lr=5e-5, 500 steps | {100*correct/total_test:.0f}% | Simplified task |

### v4 Details
- Dataset: ICBHI 2017, {len(records)} recordings, balanced to {len(train_pairs)} train
- Output: single word (normal/wheeze/crackle/both) instead of JSON
- Spectrograms: viridis colormap (more visual info than grayscale)
- Training: 500 steps, lr=5e-5 cosine, warmup=30, dropout=0.1
- Time: {train_time/60:.1f} min on {torch.cuda.get_device_name(0)}
- Loss: {result.training_loss:.4f}
""")
print("Per-label:")
for label, (c, t) in per_label.items():
    if t > 0: print(f"  {label}: {c}/{t} ({100*c/t:.0f}%)")